In [ ]:
# STRUCTURE:
# actual results, and initial stop (if never moved)
# no stop
# wide stop / max tolerable loss (20-40% range)
# 8, 10, 12.5, 15% stops
# 2, 2.5, 3 ATR stops
# maybe daily higher low and weekly higher low pivot stops, if I can figure that out, in a more advanced version

# ASSUMPTIONS:
# Entry is taken around the close of the day, so the first day is skipped when assessing R-multiples. Hence df.iloc[1:] syntax.
# On days where price gaps down below my stop, the stop is triggered at the open. Not always going to be the case
# No commissions or slippage on exit, so a round -1R loss if stop is hit intraday. Would like to fix in a future version.
# Intraday action is currently ignored. This may be an issue on days where my stop hits before a new high for the move. Will probably address with intraday price data and resampling. 

# FUTURE CONSIDERATIONS:
# Would I go about this process a different way with a larger dataset? Does pandas have a built in function, so I don't have to use the slower python loops?
# While this isn't a consideration for v1, for future versions, I'd like to consider intraday stops, slippage, commissions,etc. I want to make this professional-grade.
# At some point I plan to test trailing stops on profitable trades to see what the most effective method there would be. E.g. pivot lows, moving average, atr, percentage trailing, etc.
# Should I add some kind of drawdown calculation in a future version, so I can see what type of pullbacks I might expect and how viable wide stops actually are.
# Think about intraday entry and exit timing. E.g. what if my stop is below the low of the day, but I enter after the low of the day is set? Does it matter if entry is at the close?

# TASKS:
# Add max_r_capture_pct as an extension of false_negative function. What percentage of max_r_baseline is captured?
# Create pct_stop (consider whether catastrophe stop should be a row or column) and atr_stop functions.
# After creating pct_stop, change max_r_baseline to a catastrophe stop (e.g. 30-50%) as this is a more realistic baseline. Could keep no_stop as max_r_no_stop for additional value. 
# Should realised_r return current_r (from latest price) on open trades? What about exit_date and exit_price? Maybe leave exit_date and fill the others? Rename to last/final_price?
# Thinking it might be a good idea to use multiple functions for this. E.g. reading data, running simulation, outputting results.
# Could clean up the nested functions in stop_sim() by passing in dicts instead of repeating the same arguments. 

# ERRORS:
# Noticed minor inconsistency in csv input and output. Input has ticker as first column, output has entry_date first. Worth addressing, but not an immediate concern. Date first.
# Same inconsistency in functions and order of ticker/date return. Worth addressing these. 

In [18]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

In [12]:
# No stop function, used to calculate max_r_baseline.

def no_stop(data, ticker, entry_date, entry_price, stop_price): 
    
    max_price = entry_price

    for row_name, row in data.iloc[1:].iterrows():

        if row['High'] > max_price:
            max_price = row['High']

    max_r_baseline = (max_price - entry_price) / (entry_price - stop_price)

    return round(max_r_baseline, 2)

In [30]:
def false_negative_test(max_r, max_r_baseline, threshold):

    if max_r_baseline < 0.01:
        tail_pct = np.nan
    else: 
        tail_pct = (max_r / max_r_baseline) * 100
    
    if (max_r_baseline >= 3) and (tail_pct < threshold):
            return True, tail_pct

    return False, tail_pct

In [32]:
# Actual trade result function.

def actual_outcome(data, ticker, entry_date, entry_price, stop_price, exit_date, exit_price, max_r_baseline, threshold):

    realised_r = (exit_price - entry_price) / (entry_price - stop_price)
    max_price = entry_price
    
    date = pd.to_datetime(entry_date) + pd.DateOffset(days=1) # Possible calendar date issue to be aware of here. We want trading days...
    
    if not pd.isna(exit_date):
        for row_name, row in data.loc[date:exit_date].iterrows(): # Ideally would use iloc[1: ...] as elsewhere...

            if row['High'] > max_price:
                max_price = row['High']
    else:
        for row_name, row in data.iloc[1:].iterrows():

            if row['High'] > max_price:
                max_price = row['High']

    max_r = (max_price  - entry_price) / (entry_price - stop_price)

    false_negative = false_negative_test(max_r, max_r_baseline, threshold)

    return {'entry_date': entry_date, 
            'ticker': ticker, 
            'entry_price': entry_price,
            'stop_price': stop_price,
            'exit_date': exit_date,
            'exit_price': exit_price,
            'realised_r': round(realised_r, 2), 
            'max_r': round(max_r, 2),
            'max_r_baseline': max_r_baseline,
            'false_negative': false_negative[0],
            'tail_capture_pct': false_negative[1],
            'stop_type': 'Actual Outcome'}

In [33]:
# Initial stop result function.

def initial_stop(data, ticker, entry_date, entry_price, stop_price, max_r_baseline, threshold):

    max_price = entry_price
    realised_r = np.nan
    exit_date = np.nan
    exit_price = np.nan
    
    for row_name, row in data.iloc[1:].iterrows():

        if row['High'] > max_price:
            max_price = row['High']

        if stop_price >= row['Low']:
            if stop_price >= row['Open']:
                realised_r = (row['Open'] - entry_price) / (entry_price - stop_price)
                exit_price = row['Open']
                
            else:
                realised_r = (stop_price - entry_price) / (entry_price - stop_price)
                exit_price = stop_price
            
            exit_date = row_name.date()
            break
    
    if not pd.isna(realised_r):
        realised_r = round(realised_r, 2)

    if not pd.isna(exit_price):
        exit_price = round(exit_price, 2)

    max_r = (max_price - entry_price) / (entry_price - stop_price)

    false_negative = false_negative_test(max_r, max_r_baseline, threshold)

    return {'entry_date': entry_date, 
            'ticker': ticker, 
            'entry_price': entry_price, 
            'stop_price': stop_price, 
            'exit_date': exit_date,
            'exit_price': exit_price,
            'realised_r': realised_r, 
            'max_r': round(max_r, 2),
            'max_r_baseline': max_r_baseline,
            'false_negative': false_negative[0],
            'tail_capture_pct': false_negative[1],
            'stop_type': 'Initial Stop'}

# Edge cases and improvements to consider for later: 
# what if there's no data from df.iloc[1:]? 
# what should realised_r return if not stopped out? na value?
# what happens on a day if my stop hits before the high of the day? 

In [34]:
def stop_sim(ticker, entry_date, entry_price, stop_price, exit_date, exit_price, threshold):
    
    data = yf.download(ticker, start=entry_date, multi_level_index=False, auto_adjust=True, progress=False)

    max_r_baseline = no_stop(data, ticker, entry_date, entry_price, stop_price)

    sim_results = []
    sim_results.append(actual_outcome(data, ticker, entry_date, entry_price, stop_price, exit_date, exit_price, max_r_baseline, threshold))
    sim_results.append(initial_stop(data, ticker, entry_date, entry_price, stop_price, max_r_baseline, threshold))

    return sim_results


In [35]:
# Reading trades CSV file, then using it as input for a list of dicts, which stores trade outputs.
# Instead of looping yfinance for every trade, should I store price data in a csv? Or would that be unnecessary? Depends on speed.

trades = pd.read_csv('trades.csv')
threshold = 40
results = []

for row_name, row in trades.iterrows():
    results.extend(stop_sim(row['ticker'], row['entry_date'], row['entry_price'], row['stop_price'], row['exit_date'], row['exit_price'], threshold))


In [36]:
# List of nested dicts converted into dataframe.

results_df = pd.DataFrame(results)
results_df

,entry_date,ticker,entry_price,stop_price,exit_date,exit_price,realised_r,max_r,max_r_baseline,false_negative,tail_capture_pct,stop_type
0,2025-08-12,TSLA,340.84,320.15,2025-08-20,320.05,-1.00,0.39,7.64,True,5.149572,Actual Outcome
1,2025-08-12,TSLA,340.84,320.15,2025-08-20,320.15,-1.00,0.39,7.64,True,5.149572,Initial Stop
2,2025-08-26,STX,165.36,151.31,2025-11-21,229.72,4.58,9.34,31.50,True,29.649929,Actual Outcome
3,2025-08-26,STX,165.36,151.31,NaN,NaN,NaN,31.50,31.50,False,99.989836,Initial Stop
4,2025-08-26,CCL,31.89,29.34,2025-09-25,29.97,-0.75,0.30,0.78,False,38.163812,Actual Outcome
...,...,...,...,...,...,...,...,...,...,...,...,...
61,2026-04-08,AEIS,367.72,329.44,NaN,NaN,NaN,0.78,0.78,False,99.536487,Initial Stop
62,2026-04-08,LITE,896.23,761.51,NaN,NaN,NaN,0.47,0.47,False,100.713221,Actual Outcome
63,2026-04-08,LITE,896.23,761.51,NaN,NaN,NaN,0.47,0.47,False,100.713221,Initial Stop
64,2026-04-08,STX,495.76,436.84,NaN,NaN,NaN,1.90,1.90,False,100.162589,Actual Outcome


In [23]:
results_df.to_csv('results.csv', index=False)